# lltIntro, What It Takes to Train an LLM

LLM Training Lab, module 1. This notebook runs every step of the lab page on a CPU. No GPU is needed. Our recorded runs are in [https://github.com/genieincodebottle/aiml-companion/tree/main/courses/llm-training/runs/lltIntro](https://github.com/genieincodebottle/aiml-companion/tree/main/courses/llm-training/runs/lltIntro).

In [ ]:
# Runtime check. Run this first. It prints the fix for each problem it finds.
import platform, shutil, sys
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("FIX: transformers 5.22.1 needs Python 3.11 or later. On Colab, Runtime > Change runtime type, keep the default runtime.")
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f"RAM {ram:.1f} GB")
    if ram < 4:
        print("FIX: this notebook peaked at 1.80 GB of RAM in our run. Close other programs or use a machine with 4 GB or more.")
except ImportError:
    print("psutil not installed, RAM not checked")
free = shutil.disk_usage(".").free / 1e9
print(f"Free disk {free:.1f} GB")
if free < 2:
    print("FIX: the SmolLM2-135M download needs about 0.3 GB, plus room for the cache. Free some disk space.")
try:
    import torch
    print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
except ImportError:
    print("FIX: install PyTorch, pip install torch")
print("This lab runs on a CPU. No GPU is needed.")

Install the pinned transformers version. PyTorch is already installed on Colab.

In [ ]:
%pip install -q "transformers==5.22.1" "tokenizers==0.23.3"

## Step 1. Run the pretrained story model, then random weights

The same NumPy code the page runs in your browser.

In [ ]:
import base64
import json
import os
import urllib.request
import numpy as np

# The page fetches this file from the app. Here it comes from the repo, or from GitHub.
URL = "https://raw.githubusercontent.com/genieincodebottle/aiml-companion/main/courses/llm-training/labs/lltIntro/story-model.json"
if os.path.exists("story-model.json"):
    M = json.load(open("story-model.json"))
else:
    M = json.load(urllib.request.urlopen(URL))
vocab, H, CTX = M["vocab"], M["n_head"], M["ctx"]

def unpack(w):
    return np.frombuffer(base64.b64decode(w["b64"]), np.float16).astype(np.float32).reshape(w["shape"])

def norm(x, g):
    return x / np.sqrt((x * x).mean(-1, keepdims=True) + 1e-5) * g

def logits(W, ids):
    T = len(ids)
    x = W["tok"][ids] + W["pos"][:T]
    mask = np.triu(np.full((T, T), -1e9), 1)
    for i in range(M["n_layer"]):
        q, k, v = np.split(norm(x, W[f"n1.{i}"]) @ W[f"qkv.{i}"], 3, axis=1)
        q, k, v = (a.reshape(T, H, -1).transpose(1, 0, 2) for a in (q, k, v))
        a = q @ k.transpose(0, 2, 1) / np.sqrt(q.shape[-1]) + mask
        a = np.exp(a - a.max(-1, keepdims=True))
        y = (a / a.sum(-1, keepdims=True)) @ v
        x = x + y.transpose(1, 0, 2).reshape(T, -1) @ W[f"proj.{i}"]
        h = norm(x, W[f"n2.{i}"]) @ W[f"fc.{i}"]
        h = 0.5 * h * (1 + np.tanh(0.7978845608 * (h + 0.044715 * h ** 3)))
        x = x + h @ W[f"out.{i}"]
    return norm(x, W["nf"]) @ W["tok"].T

def loss(W, text):
    ids = [vocab.index(c) for c in text[: CTX + 1]]
    z = logits(W, ids[:-1])
    z = z - z.max(-1, keepdims=True)
    return float((np.log(np.exp(z).sum(-1)) - z[np.arange(len(ids) - 1), ids[1:]]).mean())

def write(W, prompt, n=200, seed=0):
    rng, ids = np.random.default_rng(seed), [vocab.index(c) for c in prompt]
    for _ in range(n):
        z = logits(W, ids[-CTX:])[-1].astype(np.float64) / 0.8
        p = np.exp(z - z.max())
        ids.append(int(rng.choice(len(vocab), p=p / p.sum())))
    return "".join(vocab[i] for i in ids)

trained = {k: unpack(w) for k, w in M["weights"].items()}
rng = np.random.default_rng(0)
random_init = {k: np.ones_like(w) if w.ndim == 1 else rng.normal(0, 0.02, w.shape).astype(np.float32) for k, w in trained.items()}
for name, W, n in [("Trained", trained, 200), ("Random weights", random_init, 80)]:
    print(name, "| loss", round(loss(W, M["eval_text"]), 2), "nats per character")
    print(write(W, "Once upon a time", n))
print("ln(V) =", round(float(np.log(len(vocab))), 2), "for V =", len(vocab), "characters")

## Step 2. The loss of a model that knows nothing

In [ ]:
import numpy as np

# A model that gives every token probability 1/V scores -ln(1/V) = ln(V) on any target.
for V in [63, 32_000, 49_152, 50_257]:
    print(f"V = {V:>6,}   ln(V) = {np.log(V):.2f} nats")

# Random logits with standard deviation std, loss averaged over every possible target.
rng = np.random.default_rng(0)
V, rows = 49_152, 64
for std in [0.0, 0.16, 1.0, 2.0]:
    z = rng.normal(0, std, size=(rows, V))
    m = z.max(axis=1, keepdims=True)
    lse = np.log(np.exp(z - m).sum(axis=1)) + m[:, 0]
    loss = (lse - z.mean(axis=1)).mean()
    print(f"logit std {std:.2f}   loss {loss:.2f}   ln(V) + std**2/2 = {np.log(V) + std**2 / 2:.2f}")

## Step 3. Load SmolLM2-135M and a copy with random weights

In [ ]:
import math
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

name = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(name)
trained = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
torch.manual_seed(0)
config = AutoConfig.from_pretrained(name)
random_init = AutoModelForCausalLM.from_config(config, dtype=torch.float32).eval()
print("vocab", len(tok), "| ln(V)", round(math.log(len(tok)), 2), "| init std", round(config.initializer_range, 4))
print("parameters", sum(p.numel() for p in trained.parameters()))

english = ("The old library stood at the end of a quiet street. Every morning, the librarian opened "
           "the doors at nine and watered the plants by the window. Children came after school to read "
           "stories, and older readers sat near the back with newspapers. On rainy days the reading room "
           "was full, and the smell of old paper filled the air.")
ids = tok(english, return_tensors="pt").input_ids
prompt = tok("The old library stood at the end of", return_tensors="pt")
with torch.no_grad():
    for label, model in [("trained", trained), ("random init", random_init)]:
        loss = model(ids, labels=ids).loss.item()
        out = model.generate(**prompt, max_new_tokens=30, do_sample=False, pad_token_id=tok.eos_token_id)
        text = tok.decode(out[0][prompt.input_ids.shape[1]:])
        print(label, "| loss", round(loss, 2), "on", ids.shape[1], "tokens")
        print("   ", " ".join(text.split()))

## Step 4. Compare English, Python code and Hindi

Predict first. Which text gets the lowest loss per token? Which needs the most tokens per word?

In [ ]:
import torch

hindi = ("पुराना पुस्तकालय एक शांत गली के आखिर में था। हर सुबह लाइब्रेरियन नौ बजे दरवाज़े खोलती थी "
         "और खिड़की के पास रखे पौधों को पानी देती थी। बच्चे स्कूल के बाद कहानियाँ पढ़ने आते थे, और बड़े "
         "पाठक अख़बार लेकर पीछे बैठते थे। बारिश के दिनों में पढ़ने का कमरा भरा रहता था, और हवा में "
         "पुराने कागज़ की खुशबू फैली रहती थी।")
python_code = '''def moving_average(values, window):
    if window <= 0:
        raise ValueError("window must be positive")
    result = []
    total = 0.0
    for i, v in enumerate(values):
        total += v
        if i >= window:
            total -= values[i - window]
        if i >= window - 1:
            result.append(total / window)
    return result
'''

# english, tok and trained come from step 3. Run this cell after it.
for name, text in [("English", english), ("Python code", python_code), ("Hindi", hindi)]:
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        loss = trained(ids, labels=ids).loss.item()
    n, words = ids.shape[1], len(text.split())
    # The loss is a mean over n - 1 predicted tokens, so the total is loss * (n - 1).
    print(f"{name:12} loss/token {loss:.2f}   tokens/word {n / words:.2f}   nats/word {loss * (n - 1) / words:.2f}")

## Step 5. Training compute by 6ND

In [ ]:
# Training compute by 6ND, for N parameters and D training tokens
N_smol, D_smol = 134_515_008, 2e12           # SmolLM2-135M, 2 trillion tokens per its model card
N_story, D_story = 209_408, 5_000 * 64 * 128  # our story model, 5,000 steps of 64 x 128 characters
c_smol, c_story = 6 * N_smol * D_smol, 6 * N_story * D_story
print(f"SmolLM2-135M  {c_smol:.3g} FLOPs")
print(f"story model   {c_story:.3g} FLOPs")
print(f"ratio         {c_smol / c_story:.3g}")

## Band checks

In [ ]:
# Band checks against our recorded run (lltIntro-t1-smollm2-135m, CPU, float32).
ids = tok(english, return_tensors="pt").input_ids  # step 4 reused the name ids for the last text
with torch.no_grad():
    l_trained = trained(ids, labels=ids).loss.item()
    l_random = random_init(ids, labels=ids).loss.item()
assert 3.15 < l_trained < 3.30, f"trained loss {l_trained:.3f} is outside 3.15 to 3.30. Check the model id and the English text."
assert 11.0 < l_random < 11.8, f"random-init loss {l_random:.3f} is outside 11.0 to 11.8. Check that from_config used the default initializer_range."
print("trained", round(l_trained, 2), "| random init", round(l_random, 2))
print("All band checks passed")

In [ ]:
# Peak RAM of this notebook so far
import psutil
mi = psutil.Process().memory_info()
peak = getattr(mi, "peak_wset", None)  # Windows
if peak is None:
    import resource  # Linux and macOS report ru_maxrss in KB and bytes respectively
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak = r * 1024 if sys.platform.startswith("linux") else r
print(f"Peak RAM {peak / 1e9:.2f} GB")